In [16]:
import pandas as pd

df = pd.read_csv('datasets/spotify_songs_2015_2025.csv')

In [4]:
import os
import re
import time
import tempfile
import requests
import librosa
import imageio_ffmpeg
import concurrent.futures
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm
from dotenv import load_dotenv
from itables import options as opt
from itables import show
from collections import Counter

load_dotenv()

# Register FFmpeg for Windows MP3 decoding
try:
    ffmpeg_dir = os.path.dirname(imageio_ffmpeg.get_ffmpeg_exe())
    if ffmpeg_dir not in os.environ["PATH"]:
        os.environ["PATH"] += os.pathsep + ffmpeg_dir
except Exception as e:
    print(f"Warning: Could not auto-set FFmpeg path: {e}")

In [5]:
def get_tempo_from_preview(preview_url):
    """Downloads 30s audio preview to a temp file and calculates BPM using librosa."""
    tmp_path = None
    try:
        audio_response = requests.get(preview_url, timeout=10)
        with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as tmp_file:
            tmp_file.write(audio_response.content)
            tmp_path = tmp_file.name
        
        y, sr = librosa.load(tmp_path, sr=None)
        tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
        bpm_val = float(np.atleast_1d(tempo)[0])
        return round(bpm_val, 2)
    except Exception as e:
        print(f"\n⚠️ Librosa Error on track: {e}")
        return None
    finally:
        if tmp_path and os.path.exists(tmp_path):
            try:
                os.remove(tmp_path)
            except Exception:
                pass

In [ ]:
df = pd.read_csv("spotify_songs_2015_2025.csv")

final_dataset = []

for index, row in tqdm(
    df.iterrows(), total=len(df), desc="Processing via Deezer"
):
  song_title = row["title"]
  artist_name = row["artist"]
  year = row["year"]
  streams = row["streams"]

  calculated_bpm = None
  deezer_duration = None
  deezer_genre = "Unknown"

  # Search Deezer for the Track
  deezer_query = f"{song_title} {artist_name}"
  search_url = (
      f"https://api.deezer.com/search?q={requests.utils.quote(deezer_query)}"
  )

  try:
    deezer_res = requests.get(search_url).json()
    if deezer_res.get("data"):
      track_data = deezer_res["data"][0]
      preview_url = track_data.get("preview")
      deezer_duration = track_data.get("duration")
      album_id = track_data.get("album", {}).get("id")

      #Calculate BPM from preview audio
      if preview_url:
        calculated_bpm = get_tempo_from_preview(preview_url)

      #Fetch native parent genre from Deezer's album endpoint
      if album_id:
        album_res = requests.get(
            f"https://api.deezer.com/album/{album_id}"
        ).json()
        genres_data = album_res.get("genres", {}).get("data", [])
        if genres_data:
          deezer_genre = genres_data[0].get("name", "Unknown")

  except Exception as e:
    print(f"\n❌ Deezer Error on '{song_title}': {e}")

  final_dataset.append({
      "Chart Year": year,
      "Track": song_title,
      "Artist": artist_name,
      "Streams": streams,
      "BPM": calculated_bpm,
      "Duration (s)": deezer_duration,
      "Genre": deezer_genre,
  })
    
  time.sleep(0.3)

master_df = pd.DataFrame(final_dataset)
print("\nFinished!")
print(master_df.head(10))

In [ ]:
master_df = pd.read_csv("bpm_counted.csv")


missing_mask = (
    master_df["BPM"].isna()
    | (master_df["BPM"] == "None")
    | (master_df["BPM"] == "")
)
missing_indices = master_df[missing_mask].index

print(f"Total rows in dataset: {len(master_df)}")
print(f"Found {len(missing_indices)} tracks missing BPM.\n")

if len(missing_indices) > 0:
  fixed_count = 0
  unfixable_count = 0

  for idx in tqdm(missing_indices, desc="Final BPM Resolution"):
    row = master_df.loc[idx]
    song_title = row["Track"]
    artist_name = row["Artist"]

    calculated_bpm = np.nan  # Default to NaN if calculation fails

    deezer_query = f"{song_title} {artist_name}"
    search_url = (
        f"https://api.deezer.com/search?q={requests.utils.quote(deezer_query)}"
    )

    try:
      deezer_res = requests.get(search_url).json()
      if deezer_res.get("data"):
        track_data = deezer_res["data"][0]
        preview_url = track_data.get("preview")

        if preview_url:
          tempo = get_tempo_from_preview(preview_url)
          if tempo is not None:
            calculated_bpm = float(tempo)
    except Exception as e:
      # If network drops or audio fails, it stays as NaN
      pass

    # Update master_df: assign float BPM or explicit NaN
    master_df.at[idx, "BPM"] = calculated_bpm

    if pd.notna(calculated_bpm):
      fixed_count += 1
    else:
      unfixable_count += 1

    time.sleep(0.4)

  print(
      f"\n✅ Fixed {fixed_count} tracks. Left {unfixable_count} tracks as NaN."
  )

#Clean column data types (ensure numeric conversion)
master_df["BPM"] = pd.to_numeric(master_df["BPM"], errors="coerce")
master_df["Streams"] = pd.to_numeric(master_df["Streams"], errors="coerce")

master_df.to_csv("bpm_counted.csv", index=False)
print("Saved final updated dataset to 'bpm_counted.csv'!")
print("\n--- FINAL BPM STATUS ---")
print(f"Valid BPMs:   {master_df['BPM'].notna().sum()}")
print(f"Missing (NaN): {master_df['BPM'].isna().sum()}")

In [6]:
# 1. Load your final CSV
df = pd.read_csv("bpm_counted.csv")

# 2. Configure viewer options for large datasets
opt.maxBytes = 0  # Remove size cap for 10k+ rows
opt.lengthMenu = [10, 25, 50, 100]

# 3. Render clean interactive table
show(df)

In [13]:
from collections import Counter

pd.set_option('display.max_rows', None)

# Count appearances for all artists
artist_summary = df['Artist'].value_counts().reset_index()
artist_summary.columns = ['Artist', 'Appearance Count']

master_df = pd.DataFrame(artist_summary)
master_df.to_csv("artist_count.csv", index=False)

show(artist_summary)